# Shallow Neural Networks

**IMSE 774 · Neural Networks · Week 3 · Sessions 5–6 · September 8 & 10, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/03-shallow-networks.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"

## Where the bends come from


`fig-decomposition`


In [ ]:
# Prince's parameters for figure 3.3, listed in problem 3.8 (p. 40).
phi0 = -0.23
phi  = np.array([-1.3, 1.3, 0.66])                 # phi_1, phi_2, phi_3
theta = np.array([[-0.2,  0.4],                    # theta_10, theta_11
                  [-0.9,  0.9],                    # theta_20, theta_21
                  [ 1.1, -0.7]])                   # theta_30, theta_31

x = np.linspace(0, 2, 500)
pre = theta[:, 0:1] + theta[:, 1:2] * x            # (3, 500) pre-activations
act = np.maximum(pre, 0)                           # (3, 500) activations
wgt = phi[:, None] * act                           # weighted activations
y   = phi0 + wgt.sum(axis=0)

joints = -theta[:, 0] / theta[:, 1]
colors = [ACCENT, "#1864AB", NDSU_GREEN]

fig, axes = plt.subplots(3, 3, figsize=(9.5, 7.5), sharex=True)
for d in range(3):
    axes[0, d].plot(x, pre[d], color=colors[d], lw=2)
    axes[0, d].axhline(0, color="k", lw=0.6)
    axes[0, d].set_title(rf"$\theta_{{{d+1}0}} + \theta_{{{d+1}1}}x$")
    axes[1, d].plot(x, act[d], color=colors[d], lw=2)
    axes[1, d].set_title(rf"$h_{d+1} = a[\theta_{{{d+1}0}} + \theta_{{{d+1}1}}x]$")
    axes[2, d].plot(x, wgt[d], color=colors[d], lw=2)
    axes[2, d].axhline(0, color="k", lw=0.6)
    axes[2, d].set_title(rf"$\phi_{d+1} h_{d+1}$")
    axes[2, d].set_xlabel("$x$")
for ax in axes.ravel():
    for j in joints:
        ax.axvline(j, color="gray", ls=":", lw=0.9)
    ax.set_ylim(-1.05, 1.05)
plt.tight_layout()
plt.show()

`fig-assembled`


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.axvspan(joints[0], joints[1], color=NDSU_GOLD, alpha=0.22, lw=0)
ax.plot(x, y, color="k", lw=2.5)
for j in joints:
    ax.axvline(j, color="gray", ls=":", lw=1)
    ax.plot(j, phi0 + np.sum(phi * np.maximum(theta[:, 0] + theta[:, 1] * j, 0)),
            "o", color=NDSU_GOLD, ms=7, mec="k", mew=0.7, zorder=5)
ax.set_xlabel("$x$"); ax.set_ylabel("$y$")
ax.set_ylim(-1.05, 1.05)
ax.set_title("Four linear regions joined at three joints")
plt.show()

`slope-check`


In [ ]:
def net(xv):
    """Equation (3.1) at a scalar input, with the parameters of figure 3.3."""
    return phi0 + np.sum(phi * np.maximum(theta[:, 0] + theta[:, 1] * xv, 0))

def region_slope(xv):
    active = (theta[:, 0] + theta[:, 1] * xv) > 0
    return float(np.sum(theta[active, 1] * phi[active])), active

h = 1e-6
probes = [0.25, 0.75, 1.3, 1.8]                    # one point inside each region
print(f"{'region':<8}{'x':>6}{'active units':>16}{'slope (theory)':>17}{'slope (measured)':>19}")
slopes = []
for r, xv in enumerate(probes, start=1):
    s, active = region_slope(xv)
    measured = (net(xv + h) - net(xv - h)) / (2 * h)
    slopes.append(s)
    units = ", ".join(f"h{d + 1}" for d in np.where(active)[0]) or "none"
    print(f"{r:<8}{xv:>6.2f}{units:>16}{s:>17.4f}{measured:>19.4f}")
print(f"\nshaded region: theta_11 phi_1 + theta_31 phi_3 = {theta[0, 1] * phi[0] + theta[2, 1] * phi[2]:.4f}")
print(f"problem 3.9  : slope(region 1) + slope(region 4) = {slopes[0] + slopes[3]:.4f},  slope(region 3) = {slopes[2]:.4f}")

## The Universal Approximation Theorem


`fig-approx`


In [ ]:
def target(x):
    return np.sin(3 * x) + 0.35 * x**2

xs = np.linspace(-2, 2, 1000)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.1), sharey=True)
for ax, n_regions in zip(axes, [3, 7, 20]):
    knots = np.linspace(-2, 2, n_regions + 1)
    approx = np.interp(xs, knots, target(knots))
    err = np.max(np.abs(approx - target(xs)))
    ax.plot(xs, target(xs), "--", color="gray", lw=1.8, label="target")
    ax.plot(xs, approx, color=NDSU_GREEN, lw=2, label="piecewise linear")
    ax.plot(knots, target(knots), "o", color=NDSU_GOLD, ms=4, mec="k", mew=0.5)
    ax.set_title(f"{n_regions} regions · max error {err:.3f}")
    ax.set_xlabel("$x$")
axes[0].set_ylabel("$y$")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Several outputs


`fig-shared-joints`


In [ ]:
#            joints at x = 0.25,       0.6,        1.2,        1.7
th = np.array([[-0.25, 1.0], [0.66, -1.1], [-1.2, 1.0], [1.19, -0.7]])
Phi = np.array([[ 0.1,  1.0, -0.8,  0.5,  0.9],     # phi_10 ... phi_14
                [-0.3, -0.6,  1.2,  0.4, -1.1]])    # phi_20 ... phi_24

x4 = np.linspace(0, 2, 600)
H = np.maximum(th[:, 0:1] + th[:, 1:2] * x4, 0)
Y = Phi[:, 0:1] + Phi[:, 1:] @ H
joints4 = -th[:, 0] / th[:, 1]

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(x4, Y[0], color=NDSU_GREEN, lw=2.2, label="$y_1[x]$")
ax.plot(x4, Y[1], color=ACCENT, lw=2.2, label="$y_2[x]$")
for j in joints4:
    ax.axvline(j, color="gray", ls=":", lw=1)
ax.set_xlabel("$x$"); ax.set_ylabel("output"); ax.legend()
ax.set_title("Shared hidden units force shared joint locations")
plt.show()

## Several inputs


`fig-2d`


In [ ]:
th2 = np.array([[-0.3,  1.0,  0.6],
                [ 0.2, -0.8,  1.1],
                [-0.5, -0.7, -1.0]])
ph2 = np.array([0.1, 1.0, -0.8, 0.9])

g = np.linspace(-2, 2, 300)
X1, X2 = np.meshgrid(g, g)
pre2 = [t[0] + t[1] * X1 + t[2] * X2 for t in th2]
Hs = [np.maximum(p, 0) for p in pre2]
Z  = ph2[0] + sum(w * h for w, h in zip(ph2[1:], Hs))

fig, axes = plt.subplots(1, 4, figsize=(12, 3.1))
for d, (ax, h) in enumerate(zip(axes[:3], Hs)):
    ax.imshow(h, extent=[-2, 2, -2, 2], origin="lower", cmap="viridis")
    ax.contour(X1, X2, pre2[d], levels=[0], colors="cyan", linewidths=1.8)
    ax.set_title(f"$h_{d+1}$"); ax.set_xlabel("$x_1$"); ax.grid(False)
    if d == 0:
        ax.set_ylabel("$x_2$")
axes[3].imshow(Z, extent=[-2, 2, -2, 2], origin="lower", cmap="magma")
for d in range(3):
    axes[3].contour(X1, X2, pre2[d], levels=[0], colors="cyan", linewidths=1.2, alpha=0.8)
axes[3].set_title("$y$ (weighted sum)"); axes[3].set_xlabel("$x_1$"); axes[3].grid(False)
plt.tight_layout()
plt.show()

patterns = np.stack([p > 0 for p in pre2], axis=-1).reshape(-1, 3)
unique_patterns = np.unique(patterns, axis=0)
print(f"distinct activation patterns on the plane: {len(unique_patterns)}")
for pat in unique_patterns:
    print("   active:", ", ".join(f"h{d + 1}" for d in np.where(pat)[0]) or "none")

## Regions explode with dimension


`fig-regions`


In [ ]:
from math import comb

D_units = np.arange(1, 1001)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for Di, c in zip([1, 5, 10, 50, 100], plt.cm.viridis(np.linspace(0.1, 0.9, 5))):
    regions = [sum(comb(int(D), j) for j in range(0, min(Di, int(D)) + 1)) for D in D_units]
    ax.plot(D_units, regions, color=c, lw=2, label=f"$D_i = {Di}$")
ax.plot(500, sum(comb(500, j) for j in range(0, 101)), "o", color="k", ms=6, zorder=5)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("hidden units $D$"); ax.set_ylabel("maximum linear regions $N_r$")
ax.set_title("Linear regions vs. hidden units")
ax.legend(fontsize=8)
plt.show()

n_regions = sum(comb(500, j) for j in range(0, 101))
n_params  = 500 * (100 + 1) + 1 * (500 + 1)
print(f"D = 500 hidden units, D_i = 100 inputs, D_o = 1 output")
print(f"  parameters             : {n_params:,}")
print(f"  maximum linear regions : about 10^{np.log10(float(n_regions)):.0f}")

## The General Case


`linear-collapse`


In [ ]:
Di, D, Do = 4, 32, 2
Theta, theta0 = torch.randn(D, Di), torch.randn(D)
Phi_m, phi0_v = torch.randn(Do, D), torch.randn(Do)

x_test = torch.randn(6, Di)

# Identity "activation": the composition is affine.
h_lin = x_test @ Theta.T + theta0
y_lin = h_lin @ Phi_m.T + phi0_v

# The equivalent single affine map, computed directly.
A = Phi_m @ Theta
b = Phi_m @ theta0 + phi0_v
y_affine = x_test @ A.T + b

print(f"two-layer network with identity activation, D = {D} hidden units")
print(f"max |network output - single affine map| = {float((y_lin - y_affine).abs().max()):.2e}")

## Beyond the ReLU


`fig-activations`


In [ ]:
z = np.linspace(-4, 4, 500)
acts = {
    "ReLU":       np.maximum(z, 0),
    "Leaky ReLU": np.where(z > 0, z, 0.1 * z),
    "sigmoid":    1 / (1 + np.exp(-z)),
    "tanh":       np.tanh(z),
    "SoftPlus":   np.log1p(np.exp(z)),
    "GELU":       z * 0.5 * (1 + np.tanh(np.sqrt(2 / np.pi) * (z + 0.044715 * z**3))),
}

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2), sharex=True)
for name, color in [("sigmoid", ACCENT), ("tanh", NDSU_GREEN)]:
    axes[0].plot(z, acts[name], lw=2, color=color, label=name)
axes[0].set_title("Saturating"); axes[0].legend()
for name, color in [("ReLU", NDSU_GREEN), ("Leaky ReLU", ACCENT), ("SoftPlus", "#1864AB"), ("GELU", NDSU_GOLD)]:
    axes[1].plot(z, acts[name], lw=2, color=color, label=name)
axes[1].set_title("Non-saturating on the positive side"); axes[1].legend()
for ax in axes:
    ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
    ax.set_xlabel("$z$")
axes[0].set_ylabel("$a[z]$")
plt.tight_layout()
plt.show()

## Building One in PyTorch


`pytorch-fit`


In [ ]:
def f_true(x):
    return np.sin(3 * x) + 0.35 * x**2

x_train = np.random.uniform(-2, 2, 200)
y_train = f_true(x_train) + np.random.normal(0, 0.12, x_train.shape)

X = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
Y = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

D = 12
model = nn.Sequential(
    nn.Linear(1, D),   # theta_0 and Theta: equation (3.11)
    nn.ReLU(),         # a[.]
    nn.Linear(D, 1),   # phi_0 and Phi:     equation (3.12)
)

n_params = sum(p.numel() for p in model.parameters())
print(f"hidden units D = {D},  parameters = {n_params}  "
      f"(formula D(Di + 1) + Do(D + 1) = {D * 2 + 1 * (D + 1)})")

opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
history = []
for epoch in range(3000):
    opt.zero_grad()
    loss = loss_fn(model(X), Y)
    loss.backward()
    opt.step()
    history.append(loss.item())
print(f"final training MSE = {history[-1]:.5f}")

`fig-fit`


In [ ]:
with torch.no_grad():
    xg = torch.linspace(-2.2, 2.2, 800).unsqueeze(1)
    yg = model(xg).squeeze().numpy()
    xg_np = xg.squeeze().numpy()
    W1 = model[0].weight.squeeze().numpy()
    b1 = model[0].bias.numpy()
    learned_joints = -b1 / W1
    learned_joints = learned_joints[(learned_joints > -2.2) & (learned_joints < 2.2)]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].scatter(x_train, y_train, s=10, color="gray", alpha=0.55, label="data")
axes[0].plot(xg_np, f_true(xg_np), "--", color=ACCENT, lw=1.6, label="true function")
axes[0].plot(xg_np, yg, color=NDSU_GREEN, lw=2.2, label="network")
axes[0].plot(learned_joints, np.interp(learned_joints, xg_np, yg), "o",
             color=NDSU_GOLD, ms=6, mec="k", mew=0.6, label="joints", zorder=5)
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$")
axes[0].set_title(f"{D} hidden units, {len(learned_joints)} joints in range")
axes[0].legend(fontsize=8)
axes[1].plot(history, color=NDSU_GREEN, lw=1.5)
axes[1].set_yscale("log")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("MSE")
axes[1].set_title("Training loss")
plt.tight_layout()
plt.show()

## Capacity in action


`fig-capacity`


In [ ]:
def fit_shallow(D, epochs=2500, seed=0):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(1, D), nn.ReLU(), nn.Linear(D, 1))
    o = torch.optim.Adam(m.parameters(), lr=0.05)
    for _ in range(epochs):
        o.zero_grad(); nn.MSELoss()(m(X), Y).backward(); o.step()
    return m

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.1), sharey=True)
for ax, D_try in zip(axes, [3, 12, 100]):
    m = fit_shallow(D_try)
    with torch.no_grad():
        yy = m(xg).squeeze().numpy()
        mse = nn.MSELoss()(m(X), Y).item()
    ax.scatter(x_train, y_train, s=8, color="gray", alpha=0.45)
    ax.plot(xg_np, yy, color=NDSU_GREEN, lw=2)
    ax.set_title(f"$D = {D_try}$ · training MSE {mse:.4f}")
    ax.set_xlabel("$x$")
axes[0].set_ylabel("$y$")
plt.tight_layout()
plt.show()